# Aspect 4 — Limitations of Deeper Models (Oversmoothing)

Stacking message-passing layers lets information travel farther, but too much depth
causes **oversmoothing**: node representations converge and lose their distinctiveness,
hurting downstream performance. We measure oversmoothing directly and test whether a
mitigation restores depth-robustness.

- **Model:** GraphSAGE, **mean aggregation** (per-type linear encoder → `hidden`, then
  L × `SAGEConv(mean)` via `to_hetero`).
- **Depth sweep:** L ∈ {1, 2, 4, 8, 16}.
- **Oversmoothing measures** on target-node reps (tutorial 7): **MAD** (mean pairwise
  cosine distance, → 0 when smoothed) and **mean cosine similarity** (→ 1 when smoothed).
- **Mitigations:** `none` vs **`residual`** (skip connections) vs **`dropedge`**.

All runs share the identical encoder/head, width, optimizer, loader and graph — only
**depth** and **mitigation** change.

## 1. Setup

In [ ]:
RESULTS_ONLY = False   # True → skip preprocessing/training, analyze committed metrics.csv

import json, os, subprocess, sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

_here = Path(os.path.abspath(""))
ROOT  = _here if _here.name == "aspect4" else _here / "aspect4"
os.chdir(ROOT)
PROCESSED   = ROOT / "processed"
CHECKPOINTS = ROOT / "checkpoints"
RESULTS_CSV = ROOT / "results" / "metrics.csv"
PYTHON      = sys.executable
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE, "| ROOT:", ROOT)

## 2. Datasets & Tasks

`rel-f1/driver-dnf` (small) and `rel-trial/study-outcome` (medium, clinical-trials domain). Oversmoothing is a
depth phenomenon, so the same relational HeteroData used elsewhere serves fine here;
two graphs of different size let us check the trend generalizes.

In [ ]:
TASKS = [("rel-f1", "driver-dnf"), ("rel-trial", "study-outcome")]
for dataset, task in TASKS:
    mp = PROCESSED / dataset / task / "meta.json"
    if not mp.exists():
        print(f"{dataset}/{task}: not preprocessed yet"); continue
    m = json.load(open(mp))
    print(f"{dataset}/{task}: target={m['target_node']}  "
          f"node_types={len(m['node_types'])}  edge_types={len(m['edge_types'])}")

## 3. Preprocessing

`preprocess.py` builds `train/val/test.pt` + `meta.json` with column-wise node features
and FK→PK (+ reversed) edges.

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping preprocessing.")
else:
    if all((PROCESSED / d / t / "meta.json").exists() for d, t in TASKS):
        print("Preprocessed data found — skipping.")
    else:
        r = subprocess.run([PYTHON, "preprocess.py"], cwd=ROOT)
        if r.returncode != 0:
            raise RuntimeError("preprocess.py failed.")

## 4. Model — GraphSAGE with depth & mitigation switches

`hidden=64`, mean aggregation. A per-type linear encoder gives a uniform width so
residual connections can add layer inputs to outputs. `forward_embeddings()` returns
the pre-head target representations that the oversmoothing metrics run on.

In [ ]:
import sys; sys.path.insert(0, str(ROOT))
from models import build_model, count_parameters
from train import load_meta, load_split

if (PROCESSED / "rel-f1" / "driver-dnf" / "meta.json").exists():
    meta = load_meta("rel-f1", "driver-dnf")
    data = load_split("rel-f1", "driver-dnf", "train")
    print(f"{'Mitigation':10} {'L':>3} {'Params':>10}")
    for mit in ["none", "residual", "dropedge"]:
        for L in [1, 2, 4, 8, 16]:
            m = build_model(metadata=data.metadata(), target_node_type=meta["target_node"],
                            feat_dims=meta["node_feat_dims"], num_layers=L, mitigation=mit, hidden_dim=64)
            print(f"{mit:10} {L:>3} {count_parameters(m):>10,}")
else:
    print("Preprocess first to show parameter counts.")

## 5. Experiment Configuration

**30 runs** = 2 datasets × 5 depths × 3 mitigations × 1 seed.

| Axis | Values |
|------|--------|
| Dataset/Task | rel-f1/driver-dnf, rel-trial/study-outcome |
| Depth (L) | 1, 2, 4, 8, 16 |
| Mitigation | none, residual, dropedge |

Adam (lr=1e-3), ≤100 epochs, early stopping (patience 10) on val AUPRC, NeighborLoader.

In [ ]:
DEPTHS, MITIG, SEEDS = [1,2,4,8,16], ["none","residual","dropedge"], [0]
def ckpt_path(d, t, mit, L, s):
    return CHECKPOINTS / d / t / f"sage_{mit}_L{L}_s{s}.pt"
configs = [dict(dataset=d, task=t, mitigation=mit, num_layers=L, seed=s)
           for d, t in TASKS for mit in MITIG for L in DEPTHS for s in SEEDS]
pending = [c for c in configs if not ckpt_path(c["dataset"],c["task"],c["mitigation"],c["num_layers"],c["seed"]).exists()]
print(f"Total: {len(configs)}   Done: {len(configs)-len(pending)}   Pending: {len(pending)}")

## 6. Training

In [ ]:
if RESULTS_ONLY:
    print("RESULTS_ONLY=True — skipping training.")
elif not pending:
    print("All runs complete — skipping training.")
else:
    r = subprocess.run([PYTHON, "-u", "run_experiments.py"], cwd=ROOT)
    if r.returncode != 0:
        raise RuntimeError("run_experiments.py failed.")

## 7. Results

In [ ]:
if not RESULTS_CSV.exists():
    raise FileNotFoundError("No results yet — run training first.")
df = pd.read_csv(RESULTS_CSV)
df = df.drop_duplicates(subset=["dataset","task","mitigation","num_layers","seed"], keep="last")
df["Dataset"] = df["dataset"] + "/" + df["task"]
print(f"Loaded {len(df)} rows")
df.sort_values(["Dataset","mitigation","num_layers"])[
    ["Dataset","mitigation","num_layers","test_auprc","test_auc","mad","cos_sim"]].round(4)

### Figure 1 — Oversmoothing vs depth

MAD (↓ = smoothed) and cosine similarity (↑ = smoothed) as depth grows, per mitigation.

In [ ]:
datasets = df["Dataset"].unique()
MIT_ORDER = ["none","residual","dropedge"]
colors = {"none":"#C44E52","residual":"#4C72B0","dropedge":"#55A868"}
fig, axes = plt.subplots(len(datasets), 2, figsize=(12, 4.2*len(datasets)), squeeze=False)
for r, ds in enumerate(datasets):
    sub = df[df["Dataset"]==ds]
    for c, (mcol, ylab, arrow) in enumerate([("mad","MAD (mean avg. distance)","↓ oversmoothed"),
                                             ("cos_sim","Mean cosine similarity","↑ oversmoothed")]):
        ax = axes[r][c]
        for mit in MIT_ORDER:
            s = sub[sub["mitigation"]==mit].sort_values("num_layers")
            if len(s):
                ax.plot(s["num_layers"], s[mcol], marker="o", label=mit, color=colors[mit], lw=2)
        ax.set_xscale("log", base=2); ax.set_xticks(sub["num_layers"].unique())
        ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
        ax.set_xlabel("Depth (layers)"); ax.set_ylabel(ylab)
        ax.set_title(f"{ds} — {ylab.split('(')[0].strip()}  ({arrow})", fontsize=9, fontweight="bold")
        ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)
fig.suptitle("Figure 1: Oversmoothing vs Depth", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(ROOT/"results"/"fig1_oversmoothing.png", dpi=120, bbox_inches="tight"); plt.show()

### Figure 2 — Downstream performance vs depth

In [ ]:
fig, axes = plt.subplots(1, len(datasets), figsize=(6.2*len(datasets), 4.4), squeeze=False)
for c, ds in enumerate(datasets):
    ax = axes[0][c]; sub = df[df["Dataset"]==ds]
    for mit in MIT_ORDER:
        s = sub[sub["mitigation"]==mit].sort_values("num_layers")
        if len(s):
            ax.plot(s["num_layers"], s["test_auprc"], marker="o", label=mit, color=colors[mit], lw=2)
    ax.set_xscale("log", base=2); ax.set_xticks(sub["num_layers"].unique())
    ax.get_xaxis().set_major_formatter(plt.matplotlib.ticker.ScalarFormatter())
    ax.set_xlabel("Depth (layers)"); ax.set_ylabel("Test AUPRC")
    ax.set_title(ds, fontsize=10, fontweight="bold")
    ax.legend(fontsize=8); ax.spines[["top","right"]].set_visible(False)
fig.suptitle("Figure 2: Downstream Performance vs Depth", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(ROOT/"results"/"fig2_performance.png", dpi=120, bbox_inches="tight"); plt.show()

### Figure 3 — Performance vs oversmoothing

Does representation collapse (high cosine similarity) track the drop in performance?

In [ ]:
fig, ax = plt.subplots(figsize=(7.5,5))
markers = {ds: m for ds, m in zip(datasets, ["o","s","^","D"])}
for ds in datasets:
    for mit in MIT_ORDER:
        s = df[(df["Dataset"]==ds)&(df["mitigation"]==mit)]
        if len(s):
            ax.scatter(s["cos_sim"], s["test_auprc"], color=colors[mit], marker=markers[ds],
                       s=80, edgecolor="black", alpha=0.8, zorder=3)
ax.set_xlabel("Mean cosine similarity (→ oversmoothed)"); ax.set_ylabel("Test AUPRC")
ax.set_title("Figure 3: Performance vs Oversmoothing", fontsize=12, fontweight="bold")
from matplotlib.lines import Line2D
h = [Line2D([0],[0],marker="o",color=colors[m],ls="",label=m,markersize=8) for m in MIT_ORDER]
h += [Line2D([0],[0],marker=markers[ds],color="gray",ls="",label=ds,markersize=8) for ds in datasets]
ax.legend(handles=h, fontsize=8); ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.savefig(ROOT/"results"/"fig3_perf_vs_smoothing.png", dpi=120, bbox_inches="tight"); plt.show()

### Table 1 — Full results

In [ ]:
tab = (df.sort_values(["Dataset","mitigation","num_layers"])
         [["Dataset","mitigation","num_layers","test_auprc","test_auc","mad","cos_sim","train_time_sec","num_params"]]
         .round(4))
print(tab.to_string(index=False))

## 8. Analysis & Conclusions

### What we tested
GraphSAGE (mean aggregation) at depths 1–16 on two relational tasks, measuring both
**oversmoothing** (MAD, cosine similarity of target representations) and **downstream
performance**, and comparing a plain deep stack against **residual** connections and
**DropEdge**.

### Expectations
- `none`: as depth grows, cosine similarity → 1 and MAD → 0 (representations collapse),
  and test AUPRC peaks at a shallow depth then declines.
- `residual` / `dropedge`: both should slow the collapse (keep MAD higher) and preserve
  performance at larger depths.

### Findings
*(Fill in after the run — reference Figures 1–3 and Table 1.)*
- **Oversmoothing (Fig 1):** at what depth does `none` collapse? Do the mitigations hold MAD up?
- **Performance (Fig 2):** where does each variant peak; how far does depth hurt `none`?
- **Coupling (Fig 3):** does lower cosine similarity track higher AUPRC across configs?

### Limitations & future work
- Metrics are on target-node reps for a 1000-node sample; per-type or full-graph Dirichlet
  energy is a natural extension.
- One backbone (SAGE/mean) and two mitigations; GCN/GAT and PairNorm/normalization are future work.
- `rel-f1` is small, so its downstream signal is noisier than the larger `rel-trial`.